# HSB Signal Profiler: 90-Day Analytics Report

Niniejszy notatnik wczytuje wyniki przeprofilowane przez bazę `offline_runner.py`.
Zbada on relacje MAE (Maximum Adverse Excursion), współczynnika wygranych (Win Rate) i siły predykcyjnej wszystkich wprowadzonych do HSB warunków.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#1e1e1e", "figure.facecolor": "#121212", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})
plt.rcParams['figure.figsize'] = [12, 6]

# 1. ŁADOWANIE BAZY DANYCH (Parquet)
try:
    df = pd.read_parquet("profiler_results.parquet")
    print(f"Załadowano {len(df):,} zbadanych sygnałów rynkowych.")
except FileNotFoundError:
    print("Błąd: Nie znaleziono 'profiler_results.parquet'. Odpal najpierw offline_runner.py!")


## 1. Global Win Rate & Skuteczność Konkretnych Sygnałów

In [ ]:
# Filtrujemy tylko te sygnały, które doszły do konkluzji (WIN lub LOSS, a nie PENDING/TIMEOUT)
resolved = df[df['resolution'].isin(['WIN', 'LOSS'])].copy()

global_wr = (len(resolved[resolved['resolution'] == 'WIN']) / len(resolved)) * 100 if len(resolved) > 0 else 0
print(f"Globalny Win Rate: {global_wr:.1f}%")

# Precyzyjne sprawdzenie po nazwie sygnału
results_by_signal = resolved.groupby('signal_name').apply(
    lambda x: pd.Series({
        'Total Prowadzonych': len(x),
        'Win Rate (%)': (len(x[x['resolution']=='WIN']) / len(x)) * 100,
        'Avg Confidence (%)': x['confidence'].mean(),
        'Avg MAE (Ticks)': x['mae_ticks'].mean(),
        'Avg MFE (Ticks)': x['mfe_ticks'].mean(),
    })
).reset_index().sort_values('Win Rate (%)', ascending=False)

display(results_by_signal.round(2))


## 2. MAE vs MFE Scatter Plot (Rozkład Bezpiecznych Stop Lossów)

In [ ]:
plt.figure(figsize=(10,8))
sns.scatterplot(data=resolved, x='mae_ticks', y='mfe_ticks', hue='resolution', palette={'WIN': '#10b981', 'LOSS': '#ef4444'}, alpha=0.6)
plt.axvline(x=20, color='yellow', linestyle='--', label='Próg 20 Ticks MAE')
plt.title('Maximum Adverse Excursion vs Maximum Favorable Excursion')
plt.xlabel('MAE (Zniesienie stratne w tickach przed TP)')
plt.ylabel('MFE (Ruch zyskowny tickach)')
plt.legend()
plt.show()


## 3. Optymalizacja Konfluencji (Złoty Graal)
Analizujemy, czy dorzucenie określonej konfluencji np. EMA, Liquidity, zwiększa odsetek wygranych.

In [ ]:
from collections import defaultdict

conf_stats = defaultdict(lambda: {'WINS': 0, 'TOTAL': 0})

for _, row in resolved.iterrows():
    confs = row['confluences'].split('|') if row['confluences'] else []
    for c in confs:
        if not c: continue
        conf_stats[c]['TOTAL'] += 1
        if row['resolution'] == 'WIN':
            conf_stats[c]['WINS'] += 1

confl_df = pd.DataFrame.from_dict(conf_stats, orient='index')
confl_df['Win Rate (%)'] = (confl_df['WINS'] / confl_df['TOTAL']) * 100
confl_df.sort_values('Win Rate (%)', ascending=False, inplace=True)

print("### Siła Pojedynczej Konfluencji ###")
display(confl_df[confl_df['TOTAL'] > 5].round(2)) # Filtrowanie marginesów błędu stat.
